In [1]:
import torch
import lightning
import cocodeel.dataset
import cocodeel.model
import cocodeel.posthoc_model

In [2]:
# Set default dtype, device and seed.
torch.set_default_dtype(torch.float)
torch.set_default_device("cpu")
torch.manual_seed(0)

## 1. Neural Network with Covariates

In [3]:
N = 1000
# Create data.
U = torch.randn(N, 32)
X = torch.randn(N, 2)
 # Add vector of ones at first position.
X = torch.cat((torch.ones(N, 1), X), 1)
y = (torch.matmul(X, torch.tensor([1., 1., 1.])) + 
     torch.matmul(U, 0.1 * torch.ones(32)) + 
                  torch.randn(N))
dataset = cocodeel.dataset.CovarDataset(U, X, y)
train_loader = torch.utils.data.DataLoader(dataset, batch_size=int(N/20), shuffle=True)

In [4]:
# Create model.
params = {
    "backbone": torch.nn.Linear,
    "output_func": torch.nn.Identity,
    "loss_func": torch.nn.MSELoss,
    "num_features": 32,
    "num_covars": 3,
    "backbone_params": {"in_features": 32, "out_features": 32},
    "optimizer_params": {"lr": 0.01, "weight_decay": 0.01}
}
net = cocodeel.model.CovarNeuralNetwork(**params)

In [5]:
# Train model.
trainer = lightning.Trainer(max_epochs=32)
trainer.fit(net, train_loader)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
C:\Users\Manuel Pfeuffer\anaconda3\envs\cocodeel\Lib\site-packages\lightning\pytorch\trainer\connectors\logger_connector\logger_connector.py:75: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `lightning.pytorch` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default

  | Name             | Type     | Params | Mode 
------------------------------------------------------
0 | backbone         | Linear   | 1.1 K  | train
1 | output_func      | Identity | 0      | train
2 | loss_func        | MSELoss  | 0      | train
3 | deep_predictor   | Linear   | 32     | train
4 | struct_predictor | Linear   | 3  

Epoch 31: 100%|██████████| 20/20 [00:00<00:00, 243.31it/s, v_num=16]

`Trainer.fit` stopped: `max_epochs=32` reached.


Epoch 31: 100%|██████████| 20/20 [00:00<00:00, 229.36it/s, v_num=16]


In [6]:
# Check if coefficients are close to [1, 1, 1].
net.struct_predictor.weight

Parameter containing:
tensor([[0.0668, 1.0088, 0.9836]], requires_grad=True)

## 2. Post-hoc Orthogoanlized Neural Network

In [7]:
# Train PHO model.
pho = cocodeel.posthoc_model.PostHocOrthogonalizedModel(net, train_loader)

In [8]:
# Check if coefficients are close to [1, 1, 1].
pho.model.struct_predictor.weight

Parameter containing:
tensor([[1.0323, 0.9752, 0.9740]], requires_grad=True)

## 3. Re-estimating last layer using IWLS

In [9]:
# Train IWLS model.
iwls = cocodeel.lightning_model.IWLSModel(net, train_loader)

AttributeError: module 'cocodeel.lightning_model' has no attribute 'IWLSModel'